# Oryza Vision — Perbandingan Model Hama (Google Colab)
**Satu notebook, dua model yang dilatih secara terpisah:** MobileNetV3-Large vs EfficientNet-B0. Dataset yang sama, split yang sama, konfigurasi yang sama. Hasil lama di `pest-training/` **tidak ditimpa**; output baru masuk `pest-comparison/`.

**Kelas:** `Brown_Planthopper`, `Rice_Leaf_Roller`, `Rice_Leafhopper`. Sumber: [Project-AgML/CT_PestNet_crop_pest_classification](https://huggingface.co/datasets/Project-AgML/CT_PestNet_crop_pest_classification). Notebook melakukan kurasi maksimal 400 gambar asli per kelas, split 70/15/15, 10 epoch, pemilihan checkpoint melalui **validation macro F1**, evaluasi test, lalu laporan perbandingan & confusion matrix. Tidak mencampur hama dengan penyakit.


In [ ]:
#@title 1. Install package
%pip -q install -U "datasets>=3.0,<5" huggingface_hub pillow pandas scikit-learn matplotlib tqdm

In [ ]:
#@title 2. Konfigurasi & Drive
from pathlib import Path
import os, io, json, random, hashlib, shutil, gc, time
from collections import Counter
import numpy as np, pandas as pd
from PIL import Image, ImageOps
import torch, torchvision
from torch import nn
from torchvision import models, datasets as tvdatasets, transforms
from torch.utils.data import DataLoader
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from google.colab import drive

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Oryza-Vision')
assert ROOT.is_dir(), f'Folder {ROOT} tidak ditemukan. Pastikan akun Drive benar.'
SAVE=ROOT/'pest-comparison'; SAVE.mkdir(parents=True,exist_ok=True)
LOCAL=Path('/content/oryza_pest_comparison')
RAW=LOCAL/'curated'; SPLITS=LOCAL/'splits'; OUT=LOCAL/'results'
for x in (RAW,SPLITS,OUT):x.mkdir(parents=True,exist_ok=True)

SEED=42; EPOCHS=10; BATCH_SIZE=32; IMAGE_SIZE=224; LR=1e-4
MAX_PER_CLASS=400; MIN_SIDE=96
REPO='Project-AgML/CT_PestNet_crop_pest_classification'
ARCHS=['mobilenet_v3_large','efficientnet_b0']
LABEL_MAP={'brown plant hopper':'Brown_Planthopper', 'rice leaf roller':'Rice_Leaf_Roller','rice leafhopper':'Rice_Leafhopper'}
CLASSES=sorted(LABEL_MAP.values())
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if DEVICE=='cuda':torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
print('Device:',DEVICE,'Torch:',torch.__version__,'\nClasses:',CLASSES,'\nOutput:',SAVE)

In [ ]:
#@title 3. Ambil dataset satu kali + audit kualitas + deduplikasi
# Jika proses pernah selesai pada runtime ini, gunakan curated files yang sudah ada.
MARKER=RAW/'curation_complete.json'
if MARKER.exists():
    meta=json.loads(MARKER.read_text());assert meta['classes']==CLASSES
    print('Pakai curated dataset yang sudah ada:',meta['counts'])
else:
    for c in CLASSES:
        d=RAW/c;d.mkdir(parents=True,exist_ok=True)
        for old in d.glob('*.jpg'):old.unlink()
    ds=load_dataset(REPO,split='train',streaming=True)
    if 'label' not in ds.features or 'image' not in ds.features:
        raise RuntimeError(f'Dataset columns tidak sesuai: {list(ds.features)}')
    feat=ds.features['label']
    names=list(getattr(feat,'names',[]) or [])
    print('Nama label pada dataset:',names)
    if not names:
        raise RuntimeError('Metadata ClassLabel tidak ditemukan, cek format dataset.')
    missing=set(LABEL_MAP)-set(names)
    if missing:raise RuntimeError(f'Kelas tidak ditemukan dalam dataset: {sorted(missing)}')
    ds=ds.shuffle(seed=SEED,buffer_size=3000)
    counts=Counter();rejected=Counter();seen_hashes=set()
    bar=tqdm(total=MAX_PER_CLASS*len(CLASSES),desc='Kurasi hama')
    try:
        for row in ds:
            src=feat.int2str(int(row['label'])) if isinstance(row['label'],(int,np.integer)) else str(row['label'])
            if src not in LABEL_MAP:continue
            cls=LABEL_MAP[src]
            if counts[cls]>=MAX_PER_CLASS:continue
            try:
                obj=row['image']
                if isinstance(obj,dict):
                    if obj.get('bytes') is not None:obj=Image.open(io.BytesIO(obj['bytes']))
                    elif obj.get('path'):obj=Image.open(obj['path'])
                im=ImageOps.exif_transpose(obj).convert('RGB')
                if min(im.size)<MIN_SIDE:rejected['too_small']+=1;continue
                digest=hashlib.sha256(im.resize((224,224)).tobytes()).hexdigest()
                if digest in seen_hashes:rejected['duplicate']+=1;continue
                seen_hashes.add(digest)
                path=RAW/cls/f'{counts[cls]:04d}_{digest[:12]}.jpg'
                im.save(path,'JPEG',quality=94)
                counts[cls]+=1;bar.update(1)
            except Exception as e:rejected['unreadable']+=1
            if all(counts[c]>=MAX_PER_CLASS for c in CLASSES):break
    finally:bar.close()
    if any(counts[c]<100 for c in CLASSES):
        raise RuntimeError(f'Terlalu sedikit data untuk perbandingan: {dict(counts)}. Dataset perlu diperiksa.')
    meta={'source':REPO,'seed':SEED,'classes':CLASSES,'counts':dict(counts),'rejected':dict(rejected)}
    MARKER.write_text(json.dumps(meta,indent=2))
    print('Kurasi selesai:',meta)
# Simpan curated dataset ke Drive sebagai ZIP agar split bisa diulangi dengan gambar yang identik.
ZIP=SAVE/'pest_curated_images.zip'
if not ZIP.exists():
    shutil.make_archive(str(ZIP.with_suffix('')),'zip',root_dir=RAW)
shutil.copy2(MARKER,SAVE/'curation_audit.json')
print('Curated ZIP tersimpan:',ZIP)

In [ ]:
#@title 4. Buat satu split tetap untuk kedua model (70/15/15)
if SPLITS.exists():shutil.rmtree(SPLITS)
rows=[]
for cls in CLASSES:
    files=sorted((RAW/cls).glob('*.jpg'))
    assert len(files)>=100,cls
    tr,rest=train_test_split(files,test_size=.30,random_state=SEED,shuffle=True)
    va,te=train_test_split(rest,test_size=.50,random_state=SEED,shuffle=True)
    for split,paths in [('train',tr),('val',va),('test',te)]:
        dst=SPLITS/split/cls;dst.mkdir(parents=True,exist_ok=True)
        for src in paths:
            shutil.copy2(src,dst/src.name)
            rows.append({'split':split,'class':cls,'filename':src.name,'relative_path':f'{split}/{cls}/{src.name}'})
manifest=pd.DataFrame(rows).sort_values(['split','class','filename'])
manifest.to_csv(OUT/'pest_comparison_manifest.csv',index=False)
print(pd.crosstab(manifest['class'],manifest['split']))
assert len(manifest)==len(set(manifest['filename'])),'Duplikat filename lintas split'
shutil.copy2(OUT/'pest_comparison_manifest.csv',SAVE/'pest_comparison_manifest.csv')

In [ ]:
#@title 5. Transformasi, loader dan fungsi evaluasi
TRAIN_TF=transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE,scale=(.8,1.0)),
    transforms.RandomHorizontalFlip(),transforms.RandomRotation(15),
    transforms.ColorJitter(.12,.12,.12,.04),
    transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
TEST_TF=transforms.Compose([
    transforms.Resize(256),transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
data={s:tvdatasets.ImageFolder(SPLITS/s,transform=TRAIN_TF if s=='train' else TEST_TF) for s in ['train','val','test']}
assert all(data[s].classes==CLASSES for s in data),[data[s].classes for s in data]
loaders={s:DataLoader(data[s],batch_size=BATCH_SIZE,shuffle=s=='train',num_workers=2,pin_memory=DEVICE=='cuda') for s in data}

def make_model(arch):
    if arch=='mobilenet_v3_large':
        model=models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        model.classifier[-1]=nn.Linear(model.classifier[-1].in_features,len(CLASSES))
    elif arch=='efficientnet_b0':
        model=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[-1]=nn.Linear(model.classifier[-1].in_features,len(CLASSES))
    else:raise ValueError(arch)
    return model.to(DEVICE)

def predict(model,split):
    model.eval();yt=[];yp=[]
    with torch.inference_mode():
        for imgs,targets in loaders[split]:
            logits=model(imgs.to(DEVICE));pred=logits.argmax(1).cpu().tolist()
            yt.extend(targets.tolist());yp.extend(pred)
    return np.asarray(yt),np.asarray(yp)

In [ ]:
#@title 6. Training dua model secara berurutan (checkpoint otomatis disimpan ke Drive)
# Jika notebook diulang, checkpoint model selesai dipakai kembali, bukan ditraining ulang.
summary=[]
for arch in ARCHS:
    checkpoint=SAVE/f'pest_{arch}_best.pth'
    reportfile=SAVE/f'pest_{arch}_test_metrics.json'
    historyfile=SAVE/f'pest_{arch}_history.csv'
    if checkpoint.exists() and reportfile.exists() and historyfile.exists():
        print('Sudah lengkap, skip:',arch)
        old=json.loads(reportfile.read_text())
        summary.append({'model':arch,'test_accuracy':old['accuracy'],'test_macro_f1':old['macro avg']['f1-score'],'best_val_macro_f1':float(pd.read_csv(historyfile)['val_macro_f1'].max())})
        continue
    # Pulihkan seed tiap arsitektur agar eksperimen rerun konsisten.
    torch.manual_seed(SEED)
    if DEVICE=='cuda':torch.cuda.manual_seed_all(SEED)
    model=make_model(arch)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    loss_fn=nn.CrossEntropyLoss()
    best=-1.0;history=[]
    for epoch in range(1,EPOCHS+1):
        start=time.time();model.train();running=0.0
        for xb,yb in tqdm(loaders['train'],desc=f'{arch} epoch {epoch}/{EPOCHS}',leave=False):
            xb=xb.to(DEVICE);yb=yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            out=model(xb);loss=loss_fn(out,yb)
            loss.backward();optimizer.step()
            running+=float(loss.item())*len(yb)
        yt,yp=predict(model,'val')
        val_f1=f1_score(yt,yp,average='macro',zero_division=0)
        row={'epoch':epoch,'train_loss':running/len(data['train']),
             'val_accuracy':accuracy_score(yt,yp),'val_macro_f1':val_f1,'seconds':round(time.time()-start,1)}
        history.append(row)
        print(arch,row)
        if val_f1>best:
            best=val_f1
            torch.save({'model_state':model.state_dict(),'classes':CLASSES,'architecture':arch,
                        'image_size':IMAGE_SIZE,'task':'pest','best_val_f1':best,'seed':SEED,
                        'preprocessing':{'resize':256,'center_crop':IMAGE_SIZE,
                          'mean':[.485,.456,.406],'std':[.229,.224,.225]}},checkpoint)
        pd.DataFrame(history).to_csv(historyfile,index=False)
    # Test hanya setelah pemilihan checkpoint berdasar validation
    saved=torch.load(checkpoint,map_location=DEVICE,weights_only=True)
    model.load_state_dict(saved['model_state'],strict=True)
    yt,yp=predict(model,'test')
    report=classification_report(yt,yp,labels=list(range(len(CLASSES))),target_names=CLASSES,output_dict=True,zero_division=0)
    reportfile.write_text(json.dumps(report,indent=2))
    pd.DataFrame(confusion_matrix(yt,yp,labels=list(range(len(CLASSES)))),index=CLASSES,columns=CLASSES).to_csv(SAVE/f'pest_{arch}_confusion_matrix.csv')
    print('\
',arch,'test accuracy',report['accuracy'],'macro F1',report['macro avg']['f1-score'])
    summary.append({'model':arch,'test_accuracy':report['accuracy'],'test_macro_f1':report['macro avg']['f1-score'],'best_val_macro_f1':best})
    del model,optimizer
    gc.collect()
    if DEVICE=='cuda':torch.cuda.empty_cache()
comparison=pd.DataFrame(summary)
comparison.to_csv(SAVE/'pest_model_comparison.csv',index=False)
print('\nHASIL PERBANDINGAN\n',comparison.to_string(index=False))

In [ ]:
#@title 7. Buat grafik perbandingan & confusion matrix
comparison=pd.read_csv(SAVE/'pest_model_comparison.csv')
metrics=['test_accuracy','test_macro_f1','best_val_macro_f1']
ax=comparison.set_index('model')[metrics].T.plot(kind='bar',figsize=(9,5),ylim=(0,1.05),rot=0)
ax.set_title('Oryza Vision — Perbandingan Model Hama')
ax.set_ylabel('Skor (0–1)');ax.legend(title='Architecture');plt.tight_layout()
plt.savefig(SAVE/'pest_model_comparison.png',dpi=170,bbox_inches='tight');plt.show()
for arch in ARCHS:
    cm=pd.read_csv(SAVE/f'pest_{arch}_confusion_matrix.csv',index_col=0)
    fig,ax=plt.subplots(figsize=(6,5))
    image=ax.imshow(cm.values,cmap='Blues')
    ax.set_xticks(range(len(CLASSES)),CLASSES,rotation=40,ha='right')
    ax.set_yticks(range(len(CLASSES)),CLASSES)
    ax.set_xlabel('Predicted');ax.set_ylabel('Actual');ax.set_title(arch)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):ax.text(j,i,str(cm.iloc[i,j]),ha='center',va='center')
    plt.colorbar(image,ax=ax);plt.tight_layout()
    plt.savefig(SAVE/f'pest_{arch}_confusion_matrix.png',dpi=160,bbox_inches='tight');plt.show()

In [ ]:
#@title 8. Pilih model berdasar validation Macro F1 (bukan test set)
comparison=pd.read_csv(SAVE/'pest_model_comparison.csv')
best=comparison.sort_values(['best_val_macro_f1','model'],ascending=[False,True]).iloc[0]
selected={'selected_model':str(best['model']),'selection_metric':'best_val_macro_f1',
          'best_val_macro_f1':float(best['best_val_macro_f1']),
          'classes':CLASSES,'dataset':REPO,'seed':SEED,'image_size':IMAGE_SIZE,
          'note':'Model dipilih berdasar validation; test score hanya evaluasi final. Benchmark latensi dan RAM Android diperlukan sebelum deployment.'}
(SAVE/'pest_selected_model.json').write_text(json.dumps(selected,indent=2))
(SAVE/'pest_class_names.json').write_text(json.dumps(CLASSES,indent=2))
print('MODEL PILIHAN:',selected['selected_model'])
print('Saved files:',*[p.name for p in sorted(SAVE.iterdir()) if p.is_file()],sep='\n- ')
print('Selesai. Tidak mengubah checkpoint lama di pest-training/.')